# OGD Polytope Gateway: end-to-end test

**Status as of 14 October 2026 — draft for project review; not ready to merge.**

An end-to-end retrieval with earthkit has succeeded through the temporary CloudFront address. This notebook records the current project state and provides a small example for testers.

| Item | Current state |
|---|---|
| End-to-end access | Successfully tested through CloudFront to Polytope at CSCS |
| Address | Temporary CloudFront hostname; final service address pending |
| Authentication | A gateway token has been issued using the gateway token-creation script |
| Anonymous access | Successfully tested, then disabled during testing to avoid unrestricted public access |

Requests pass through CloudFront, API Gateway and the Lambda proxy to Polytope at CSCS. The proxy supplies the upstream Polytope credentials; testers use their issued **gateway token**.

## Installation

Run these commands in a terminal from your project folder:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m ensurepip --upgrade
python -m pip install ipykernel earthkit-data polytope-client covjsonkit
python -m ipykernel install --user --name gateway-demo --display-name "Python (Gateway demo)"
```

In VS Code, choose **Select Kernel → Select Another Kernel → Jupyter Kernel → Python (Gateway demo)**. If it is missing, run **Developer: Reload Window** from the Command Palette (`Ctrl+Shift+P`). For SSH or container sessions, run the commands in that same environment.


## Configure gateway access

The hostname below is the temporary development endpoint. Replace it with the final service address when available. Enter the gateway token issued for testing (`pt_...`) at the hidden prompt. Blank input stops the example; there is no anonymous fallback.

In [1]:
import os
from getpass import getpass
import earthkit.data as ekd

# Temporary gateway address; replace when the final address is ready.
ADDRESS = "d2cbt52revkmkg.cloudfront.net"
COLLECTION = "mchgj"

# Enter the token issued for testing. Anonymous access is currently disabled.
token = getpass("Gateway token: ").strip()
if not token:
    raise ValueError("A gateway token is required.")

# Clear inherited settings that could change the authentication method.
for name in ("POLYTOPE_USER_EMAIL", "POLYTOPE_USERNAME", "POLYTOPE_PASSWORD"):
    os.environ.pop(name, None)

auth_options = {"user_key": token}

Gateway token:  ········


## Define the request

Retrieve **2-metre temperature** from **ICON-CH2-EPS**, for ensemble members 1–10 at a point near Zurich. Coordinates are longitude, then latitude. This follows the gateway example's timeseries request and returns CoverageJSON for conversion to xarray.

`date="0"` selects today and `time="0000"` the 00 UTC run. Real-time storage holds recent forecasts only; reruns depend on run availability. The requested lead-time range is 0–48 hours, but a newly available run may contain fewer steps.


In [4]:
REQUEST = {
    "class": "od",
    "model": "icon-ch2-eps",
    "stream": "enfo",
    "type": "pf",
    "levtype": "sfc",
    "expver": "0001",
    "number": list(range(1, 11)),
    "param": "500011",  # 2-metre temperature (t_2m).
    "date": "0",
    "time": "0000",
    "timespan": "none",
    "feature": {
        "type": "timeseries",
        "points": [[8.54, 47.45]],
        "time_axis": "step",
        "range": {"start": 0, "end": 48},
        "axes": ["longitude", "latitude"],
    },
}


## Retrieve and inspect the result

A successful run returns an `xarray.Dataset` containing `t_2m`. Inspect the coordinates to see which forecast times and nearest model-grid point were returned.


In [3]:
ds = ekd.from_source(
    "polytope", COLLECTION, REQUEST,
    address=ADDRESS,
    **auth_options,
    stream=False,
).to_xarray()

if "t_2m" not in ds or ds["t_2m"].size == 0:
    raise RuntimeError("No temperature data returned; check the requested run and availability.")
print("Retrieval successful. Dimensions:", dict(ds.sizes))
ds


2026-10-14 18:11:50 - INFO - Sending request...
{'request': 'class: od\n'
            "date: '0'\n"
            "expver: '0001'\n"
            'feature:\n'
            '  axes:\n'
            '  - longitude\n'
            '  - latitude\n'
            '  points:\n'
            '  - - 8.54\n'
            '    - 47.45\n'
            '  range:\n'
            '    end: 48\n'
            '    start: 0\n'
            '  time_axis: step\n'
            '  type: timeseries\n'
            'levtype: sfc\n'
            'model: icon-ch2-eps\n'
            'number:\n'
            '- 1\n'
            '- 2\n'
            '- 3\n'
            '- 4\n'
            '- 5\n'
            '- 6\n'
            '- 7\n'
            '- 8\n'
            '- 9\n'
            '- 10\n'
            "param: '500011'\n"
            'stream: enfo\n'
            "time: '0000'\n"
            'timespan: none\n'
            'type: pf\n',
 'verb': 'retrieve'}
2026-10-14 18:11:51 - INFO - Request accepted. Please poll ./fee8d123-a

Retrieval successful. Dimensions: {'latitude': 1, 'longitude': 1, 'levelist': 1, 'number': 10, 'datetime': 1, 't': 4}


<xarray.Dataset> Size: 536B
Dimensions:    (latitude: 1, longitude: 1, levelist: 1, number: 10,
                datetime: 1, t: 4)
Coordinates:
  * latitude   (latitude) float64 8B 47.46
  * longitude  (longitude) float64 8B 8.549
  * levelist   (levelist) int64 8B 0
  * number     (number) int64 80B 1 2 3 4 5 6 7 8 9 10
  * datetime   (datetime) <U20 80B '2026-10-14T00:00:00Z'
  * t          (t) datetime64[ns] 32B 2026-10-14 ... 2026-10-14T03:00:00
Data variables:
    t_2m       (latitude, longitude, levelist, number, datetime, t) float64 320B ...
Attributes:
    class:          od
    Forecast date:  2026-10-14T00:00:00Z
    expver:         0001
    levtype:        sfc
    model:          icon-ch2-eps
    number:         1
    stream:         enfo
    timespan:       none
    type:           pf
    levelist:       0

## Anonymous access: tested approach, currently disabled

Anonymous retrieval was tested before this path was disabled.


```python
# Reference only: anonymous access is currently disabled.
anonymous_auth = {
    "username": "anonymous",
    "password": "anonymous",
}

ds_anonymous = ekd.from_source(
    "polytope", COLLECTION, REQUEST,
    address=ADDRESS,
    **anonymous_auth,
    stream=False,
).to_xarray()
```


## Before merging

- Replace the temporary CloudFront hostname with the final service address.
- Repeat the token-authenticated end-to-end retrieval against that address.
- Confirm that requests without a valid gateway token remain rejected during testing.
